# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umarfarukh786/FlyRank-task1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

This cell builds the same modeling-ready matrix used later in the refresh-ranking lane. Numeric fields are coerced and missing numeric values are filled with zero only after preserving the source missingness audit; categorical values use an explicit `unknown` category and one-hot encoding. The target is stored separately. IDs and trend-derived fields are never passed into the feature matrix.

In [5]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES

RAW_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Starter dataset not found: {RAW_PATH}")

raw = pd.read_csv(RAW_PATH)
raw["is_declining_label"] = raw["trend_direction"].astype(str).str.lower().eq("down").astype(int)
for column in ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d"]:
    source_column = column.replace("log_", "")
    raw[column] = np.log1p(pd.to_numeric(raw[source_column], errors="coerce").fillna(0).clip(lower=0))

numeric_features = [column for column in MODEL_NUMERIC_FEATURES if column in raw.columns]
categorical_features = [column for column in MODEL_CATEGORICAL_FEATURES if column in raw.columns]
numeric_frame = raw[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
numeric_missing = numeric_frame.isna().mean().sort_values(ascending=False)
numeric_frame = numeric_frame.fillna(0)
categorical_frame = raw[categorical_features].fillna("unknown").astype(str)
encoded_categorical = pd.get_dummies(
    categorical_frame,
    prefix=categorical_features,
    dummy_na=False,
    dtype=float,
)
feature_matrix = pd.concat(
    [numeric_frame.reset_index(drop=True), encoded_categorical.reset_index(drop=True)],
    axis=1,
)
target = raw["is_declining_label"].astype(int)
feature_names = list(feature_matrix.columns)

print(f"Rows: {len(raw):,}")
print(f"Feature columns after encoding: {len(feature_names):,}")
print(f"Numeric source features: {len(numeric_features)} | categorical source features: {len(categorical_features)}")
print(f"Target positive rate: {target.mean():.3f}")
print("Highest source numeric missingness before fill:")
print(numeric_missing.head(8).round(3).to_string())
assert "content_id" not in feature_names
assert "client_id" not in feature_names
assert "trend_direction" not in feature_names
assert "trend_pct" not in feature_names
print("Feature-vector construction and basic exclusion checks: passed")

Rows: 30,000
Feature columns after encoding: 52
Numeric source features: 18 | categorical source features: 8
Target positive rate: 0.542
Highest source numeric missingness before fill:
char_count          0.257
word_count          0.257
competition         0.082
search_volume       0.082
cpc                 0.082
scroll_rate         0.004
log_clicks_90d      0.000
log_sessions_90d    0.000
Feature-vector construction and basic exclusion checks: passed


## 2. Feature notes: meaning, missing, categorical, available-when?

The feature vector contains snapshot signals that are intended to be available at review time: traffic totals, search context, content size, age/freshness, position, rates, and safe categorical tiers. Numeric missing values are filled after their missingness is measured; categorical blanks become `unknown`. This is a modeling convenience, not evidence that a missing metric was truly zero. The `trend_direction`/`trend_pct` label source, IDs, current comparison windows, and product decision outputs remain outside the matrix.

In [6]:
feature_notes = pd.DataFrame([
    {
        "feature_group": "numeric",
        "examples": "traffic totals, rates, position, age, freshness, search context",
        "missing_handling": "measure missingness, then numeric fill with 0 for the starter matrix",
        "available_when": "snapshot/review time; temporal alignment must be checked for a future label",
    },
    {
        "feature_group": "categorical",
        "examples": "content type, intent, age/freshness and visibility tiers",
        "missing_handling": "fill blank categories with explicit unknown before one-hot encoding",
        "available_when": "snapshot/review time",
    },
    {
        "feature_group": "engineered numeric",
        "examples": "log1p impressions, clicks, sessions, and AI sessions",
        "missing_handling": "coerce invalid values, replace non-finite values, then log1p nonnegative totals",
        "available_when": "snapshot/review time, subject to the label-window audit",
    },
])
display(feature_notes)
print("Feature availability note: this starter slice is cross-sectional; a future-looking label needs a separate time-aware feature build.")

,feature_group,examples,missing_handling,available_when
0,numeric,"traffic totals, rates, position, age, freshnes...","measure missingness, then numeric fill with 0 ...",snapshot/review time; temporal alignment must ...
1,categorical,"content type, intent, age/freshness and visibi...",fill blank categories with explicit unknown be...,snapshot/review time
2,engineered numeric,"log1p impressions, clicks, sessions, and AI se...","coerce invalid values, replace non-finite valu...","snapshot/review time, subject to the label-win..."


Feature availability note: this starter slice is cross-sectional; a future-looking label needs a separate time-aware feature build.


## 3. The leakage hunt

The audit attacks four risks: label-derived columns, future or overlapping windows, pseudonymous IDs, and product-decision outputs. A clean result means the explicit suspect columns are absent; the snapshot’s lack of row-level dates still leaves a temporal-alignment limitation that must be disclosed rather than silently called safe.

In [7]:
forbidden_feature_columns = {
    "content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label",
    "health_score", "priority_score", "action_type", "refresh_tier",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
}
used_source_features = set(numeric_features + categorical_features)
explicit_leaks = used_source_features.intersection(forbidden_feature_columns)
leakage_audit = pd.DataFrame([
    {
        "risk": "label-derived feature",
        "status": "PASSED" if not used_source_features.intersection({"trend_direction", "trend_pct", "is_declining_label"}) else "FAILED",
        "evidence": sorted(used_source_features.intersection({"trend_direction", "trend_pct", "is_declining_label"})) or "none",
    },
    {
        "risk": "pseudonymous ID used as feature",
        "status": "PASSED" if not used_source_features.intersection({"content_id", "client_id"}) else "FAILED",
        "evidence": sorted(used_source_features.intersection({"content_id", "client_id"})) or "none",
    },
    {
        "risk": "product decision output used as feature",
        "status": "PASSED" if not used_source_features.intersection({"health_score", "priority_score", "action_type", "refresh_tier"}) else "FAILED",
        "evidence": sorted(used_source_features.intersection({"health_score", "priority_score", "action_type", "refresh_tier"})) or "none",
    },
    {
        "risk": "comparison window overlaps proxy-label logic",
        "status": "PASSED" if not used_source_features.intersection({"impressions_last_30d", "clicks_last_30d", "sessions_last_30d", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"}) else "FAILED",
        "evidence": sorted(used_source_features.intersection({"impressions_last_30d", "clicks_last_30d", "sessions_last_30d", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"})) or "none",
    },
    {
        "risk": "strict future-window alignment",
        "status": "REVIEW REQUIRED",
        "evidence": "starter CSV has no row-level dates; current 90d snapshot fields cannot prove separation from a future label",
    },
])
display(leakage_audit)
assert not explicit_leaks

# Deliberate adversarial test: a label copy must be rejected by the same guard.
with_deliberate_leak = set(used_source_features) | {"is_declining_label"}
assert with_deliberate_leak.intersection(forbidden_feature_columns)
print("Adversarial label-column test: caught")
print("Explicit leakage checks: passed; temporal alignment: review required for any future-label redesign.")

,risk,status,evidence
0,label-derived feature,PASSED,none
1,pseudonymous ID used as feature,PASSED,none
2,product decision output used as feature,PASSED,none
3,comparison window overlaps proxy-label logic,PASSED,none
4,strict future-window alignment,REVIEW REQUIRED,starter CSV has no row-level dates; current 90...


Adversarial label-column test: caught
Explicit leakage checks: passed; temporal alignment: review required for any future-label redesign.


## 4. What I excluded and why

The exclusions are deliberate: trend fields define the proxy label; IDs are grouping keys; current/previous comparison fields are tied to the trend calculation and could overlap a future outcome; and product scores or flags would teach the model to imitate an existing decision rather than learn observable content/search relationships.

In [8]:
excluded = pd.DataFrame([
    {
        "field": "trend_direction / trend_pct / is_declining_label",
        "reason": "label-derived fields; including them would reveal the answer",
    },
    {
        "field": "content_id / client_id",
        "reason": "pseudonymous identifiers; grouping, joining, and client-holdout splitting only",
    },
    {
        "field": "impressions_last_30d / clicks_last_30d / sessions_last_30d",
        "reason": "current comparison window contributes to the trend label",
    },
    {
        "field": "impressions_prev_30d / clicks_prev_30d / sessions_prev_30d",
        "reason": "used in the proxy trend comparison; keep out of this baseline feature vector",
    },
    {
        "field": "health_score / priority_score / action_type / refresh_tier",
        "reason": "product-decision outputs; using them would reproduce an existing rule",
    },
    {
        "field": "provider_used / model_used",
        "reason": "generation metadata is outside the observable content/search signal for this lane",
    },
])
display(excluded)
print(f"Excluded field groups documented: {len(excluded)}")
print("Privacy check: no client names, domains, URLs, private queries, or credentials are loaded.")

,field,reason
0,trend_direction / trend_pct / is_declining_label,label-derived fields; including them would rev...
1,content_id / client_id,"pseudonymous identifiers; grouping, joining, a..."
2,impressions_last_30d / clicks_last_30d / sessi...,current comparison window contributes to the t...
3,impressions_prev_30d / clicks_prev_30d / sessi...,used in the proxy trend comparison; keep out o...
4,health_score / priority_score / action_type / ...,product-decision outputs; using them would rep...
5,provider_used / model_used,generation metadata is outside the observable ...


Excluded field groups documented: 6
Privacy check: no client names, domains, URLs, private queries, or credentials are loaded.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.